In [1]:
import os
import sys

sys.path.append("/home/koledjoz/Ex2VecExtended")

In [2]:
import torch
import pandas as pd
from tqdm import tqdm
import numpy as np

from src.models.optimized.extendedBase import Ex2VecExtendedBaseFast

In [3]:
df = pd.read_parquet('../../../../sorted_data.parquet')


In [4]:
max_history = 3500
user_count = df['user_id'].max()
item_count = df['track_id'].max()

In [5]:
time_history = np.zeros((user_count+1, max_history), dtype=int)
item_history = np.zeros((user_count+1, max_history), dtype=int)

for user in tqdm(df['user_id'].unique()):
    tmp = df[df['user_id'] == user]
    item_history[user, :len(tmp)] = tmp['track_id'].to_numpy()
    time_history[user, :len(tmp)] = tmp['ts'].to_numpy()

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13209/13209 [08:05<00:00, 27.20it/s]


In [6]:
# lets remove some from the history based on our test thingies
import json

with open('../../../../split_data/test/test_dict.json', 'r') as f:
    data = json.load(f)





In [7]:
for user, items in data.items():
    np.put(item_history[int(user), :], items, [0,0])

In [8]:
from torch.utils.data import Dataset

class Ex2VecDataset(Dataset):
    def __init__(self, times, users, items):
        self.times = times
        self.users = users
        self.items = items

    def __len__(self):
        return len(self.times)

    def __getitem__(self, idx):
        return self.times[idx], self.users[idx], self.items[idx]

In [9]:
import torch
from torch.utils.data import DataLoader

data = Ex2VecDataset(df['ts'].to_numpy(), df['user_id'].to_numpy(), df['track_id'].to_numpy())

batch_size = 2**16

# train_dataloader = DataLoader(data, batch_size=batch_size, shuffle=True)

device = 'cuda'
config = {'n_users': user_count, 'n_items': item_count, 'latent_d': 64}


model = Ex2VecExtendedBaseFast(config)
model.initialize_histories(torch.tensor(item_history).to(device), torch.tensor(time_history).to(device))
model.to(device)


Ex2VecExtendedBaseFast(
  (user_lamb): Embedding(13210, 1)
  (user_bias): Embedding(13210, 1)
  (item_bias): Embedding(3028, 1)
  (embedding_user): Embedding(13210, 64)
  (embedding_item): Embedding(3028, 64)
  (logistic): Sigmoid()
)

In [10]:
model.to(device)

log_every = 100

criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

use_amp = bool(device == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

global_step = 0
history = {"train_loss": [], "val_loss": []}

for epoch in range(100):
    model.train()
    epoch_loss = 0.0
    n = 0
    train_dataloader = DataLoader(data, batch_size=batch_size, shuffle=True)
    for step, batch in enumerate(tqdm(train_dataloader)):
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(batch[0].to(device), batch[1].to(device))
            loss = criterion(outputs, batch[2].to(device))

        # Backprop (AMP-aware)
        scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()

        bs = batch[2].shape[0] if torch.is_tensor(batch[2]) and batch[2].ndim > 0 else 1
        epoch_loss += float(loss.item()) * bs
        n += bs
        global_step += 1

        # if (step % log_every == 0):
        #     lr = optimizer.param_groups[0]["lr"]
        #     avg_loss = epoch_loss / max(n, 1)
        #     print(
        #         f"[epoch {epoch}/100] "
        #         f"step {step}/{len(train_dataloader)} "
        #         f"loss={avg_loss:.4f} lr={lr:.2e} "
        #     )
    train_loss = epoch_loss / max(n, 1)
    history["train_loss"].append(train_loss)


ckpt = {
    "model_state_dict": model.state_dict(),
    "config": vars(config),
}
ckpt["optimizer_state_dict"] = optimizer.state_dict()

torch.save(ckpt, './extendedBase.pt')
print(f"Saved final checkpoint to: original.pt")



/tmp/ipykernel_1902802/1600455498.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
  0%|                                                                                                                                                                                    | 0/255 [00:00<?, ?it/s]/tmp/ipykernel_1902802/1600455498.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
  0%|                                                                                                                                                                                    | 0/255 [00:04<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 48448.00 GiB. GPU 0 has a total capacity of 31.73 GiB of which 20.09 GiB is free. Including non-PyTorch memory, this process has 11.64 GiB memory in use. Of the allocated memory 11.09 GiB is allocated by PyTorch, and 183.15 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)